In [1]:
import pandas as pd

In [39]:
results = pd.read_csv('data/dxy_training/articles_analyzed_2.csv')
y = pd.read_csv('data/dxy_training/dxy_critical_tag.csv')

In [30]:
# results.columns
# # 'event_reasoning', 'impact_confidence', 'impact_reasoning'
results['published'] = pd.to_datetime(pd.to_datetime(results['published']).dt.date)
results["dir_num"] = results["dxy_direction"].map({"DOWN": -1, "NEUTRAL": 0, "UP": 1})
results["mag_num"] = results["dxy_magnitude"].map({"LOW": 1, "MEDIUM": 2, "HIGH": 3})

results[['title', 'published', 'event_name', 'dir_num', 'mag_num']].sort_values(
    'published'
)

,title,published,event_name,dir_num,mag_num
0,Powell sees tariffs raising inflation and says...,2025-04-04,Dovish Fed Guidance,-1,2
1,Tariff rates Trump ascribes to other countries...,2025-04-04,US Trade Surplus/Deficit Narrows,1,2
2,"U.S. payrolls rose by 228,000 in March, but un...",2025-04-04,Strong US Employment (NFP),1,2
3,"Dow drops 2,200 points Friday, S&P 500 loses 1...",2025-04-04,US Treasury Yield Spike,-1,2
4,"Congress has power over tariffs, but stopping ...",2025-04-04,US Fiscal Stimulus,-1,2
5,"Trump open to tariff negotiations, contradicti...",2025-04-04,US Fiscal Stimulus,1,2
6,Trump tariffs could make an already challengin...,2025-04-04,US Treasury Yield Spike,-1,2
7,President Donald Trump says Fed Chair Powell s...,2025-04-04,Dovish Fed Guidance,-1,2
8,â€˜The damage is doneâ€™: Trumpâ€™s tariffs pu...,2025-04-11,US Treasury Yield Spike,-1,2
9,Trump trade war escalates as China raises reta...,2025-04-11,US Trade Deficit Widens,-1,2


In [22]:
results['dir_x_mag'] = results['dir_num'] * results['mag_num']
res_agg = results.groupby('published').sum()[['dir_num', 'dir_x_mag']].reset_index()

In [24]:
april_y = y[pd.to_datetime(y['Date']) >= '2025-04-01']
april_y = april_y[pd.to_datetime(y['Date']) <= '2025-04-26']
april_y = april_y.drop(columns=['Unnamed: 0'])
april_y.reset_index(drop=True, inplace=True)
april_y['Date'] = pd.to_datetime(april_y['Date'])


/var/folders/7n/r66735gn7xx2tdk0357s_k_m0000gn/T/ipykernel_39688/4208705860.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  april_y = april_y[pd.to_datetime(y['Date']) <= '2025-04-26']


In [25]:
april_y.merge(res_agg, left_on="Date", right_on="published", how="left")

,Date,Close,pct_change,category,published,dir_num,dir_x_mag
0,2025-04-01,104.260002,0.000480,0.0,NaT,NaN,NaN
1,2025-04-02,103.809998,-0.004316,0.0,NaT,NaN,NaN
2,2025-04-03,102.070000,-0.016761,-1.0,NaT,NaN,NaN
3,2025-04-04,103.019997,0.009307,1.0,2025-04-04,2.0,6.0
4,2025-04-07,103.260002,0.002330,0.0,2025-04-07,2.0,4.0
5,2025-04-08,102.959999,-0.002905,0.0,NaT,NaN,NaN
6,2025-04-09,102.900002,-0.000583,0.0,NaT,NaN,NaN
7,2025-04-10,100.870003,-0.019728,-1.0,NaT,NaN,NaN
8,2025-04-11,99.779999,-0.010806,-1.0,2025-04-11,-1.0,-3.0
9,2025-04-14,99.639999,-0.001403,0.0,2025-04-14,0.0,0.0


missed 5 critical days, captured 2 accurately, 1 false positive

only capturing 3 out of 8 critical days

In [43]:
df_res = pd.read_csv('data/dxy_training/cnbc_apr2_4_gemini_results_v0.csv')
df_res.columns

Index(['id', 'title', 'link', 'published', 'source', 'query',
       'priority_weight', 'article_date', 'prediction_date', 'fetched_at',
       'title_len', 'is_relevant', 'content_type', 'event_number',
       'event_name', 'event_tier', 'event_tier_label', 'vs_consensus',
       'is_event_vs_opinion', 'classification_confidence', 'is_critical',
       'criticality_level', 'tier_override', 'override_reason',
       'data_point_referenced', 'reported_value', 'processed_at'],
      dtype='str')

In [56]:
df_res[df_res['is_relevant']==True][['published', 'title','is_relevant', 'content_type', 'event_number',
       'event_name', 'event_tier', 'event_tier_label', 'vs_consensus',
       'is_event_vs_opinion', 'classification_confidence', 'is_critical',
       'criticality_level', 'tier_override', 'override_reason',
       'data_point_referenced', 'reported_value']]

,published,title,is_relevant,content_type,event_number,event_name,event_tier,event_tier_label,vs_consensus,is_event_vs_opinion,classification_confidence,is_critical,criticality_level,tier_override,override_reason,data_point_referenced,reported_value
0,"Fri, 04 Apr 2025 07:00:00 GMT","US Stock Market Highlights: Dow slides 1,600 p...",True,hard_news,24.0,Risk-Off Shock (flight to USD safety),2.0,Tier 2 — Secondary Structural (Risk/Flows/Fiscal),not-applicable,True,high,True,high,True,The article describes immediate and significan...,True,2.1%
1,"Fri, 04 Apr 2025 07:00:00 GMT",Trump throws half a punch: 27% reciprocal tari...,True,hard_news,19.0,Trade Tariff / Policy Shock,3.0,Tier 3 — Geopolitical & Sentiment,not-applicable,True,high,True,medium,True,The article describes a significant escalation...,True,26%-27%
2,"Fri, 04 Apr 2025 07:00:00 GMT","Trump tariffs fallout: China retaliates, Vietn...",True,hard_news,19.0,Trade Tariff / Policy Shock,3.0,Tier 3 — Geopolitical & Sentiment,not-applicable,True,high,True,high,True,The article describes immediate and significan...,True,34%
3,"Fri, 04 Apr 2025 07:00:00 GMT",Powell sees tariffs raising inflation and says...,True,hard_news,30.0,Trade Policy Escalation Signal (new tariffs or...,3.0,Tier 3 — Geopolitical & Sentiment,not-applicable,False,high,False,low,False,NaN,True,10%
4,"Fri, 04 Apr 2025 07:00:00 GMT",Tariff rates Trump ascribes to other countries...,True,hard_news,19.0,Trade Tariff / Policy Shock,3.0,Tier 3 — Geopolitical & Sentiment,not-applicable,False,high,False,low,False,NaN,True,67%
12,"Fri, 04 Apr 2025 07:00:00 GMT","Klarna, StubHub delay IPOs as Trump's tariffs ...",True,hard_news,28.0,Other / Mixed,4.0,Tier 4 — Commodity/Technical or Ambiguous,not-applicable,False,low,False,low,False,NaN,False,NaN


In [48]:
y[y['category'] !=0].reset_index(drop=True).loc[20:]

,Unnamed: 0,Date,Close,pct_change,category
20,71,2025-04-15,100.220001,0.005821,1.0
21,72,2025-04-16,99.379997,-0.008382,-1.0
22,74,2025-04-21,98.279999,-0.011069,-1.0
23,75,2025-04-22,98.919998,0.006512,1.0
24,76,2025-04-23,99.839996,0.009300,1.0
25,77,2025-04-24,99.290001,-0.005509,-1.0
26,82,2025-05-01,100.250000,0.007842,1.0
27,85,2025-05-06,99.239998,-0.005910,-1.0
28,87,2025-05-08,100.639999,0.010340,1.0
29,89,2025-05-12,101.790001,0.014451,1.0


In [46]:
y.loc[230:272]

,Unnamed: 0,Date,Close,pct_change,category
230,230,2025-12-01,99.410004,-0.000503,0.0
231,231,2025-12-02,99.360001,-0.000503,0.0
232,232,2025-12-03,98.849998,-0.005133,-1.0
233,233,2025-12-04,98.989998,0.001416,0.0
234,234,2025-12-05,98.989998,0.000000,0.0
235,235,2025-12-08,99.089996,0.001010,0.0
236,236,2025-12-09,99.220001,0.001312,0.0
237,237,2025-12-10,98.790001,-0.004334,0.0
238,238,2025-12-11,98.349998,-0.004454,0.0
239,239,2025-12-12,98.400002,0.000508,0.0


In [36]:
import pandas as pd

# df_ft = pd.read_csv('data/dxy_training/gemeni/cnbc_apr_2_4_full_text.csv')
df_ft = pd.read_csv('data/dxy_training/claude/april_cln.csv')

In [37]:
df_ft.shape

(293, 13)

In [8]:
EVENT_NAMES = {
    0:  "Irrelevant",
    1:  "Fed Rate Hike",
    2:  "Fed Rate Cut",
    3:  "Hawkish Pivot / Surprise",
    4:  "Dovish Pivot / Surprise",
    5:  "Fed QE / Balance Sheet Expansion",
    6:  "Fed QT / Tapering Announcement",
    7:  "CPI/PCE Above Consensus",
    8:  "CPI/PCE Below Consensus",
    9:  "Inflation In-Line",
    10: "NFP/Jobs Beat",
    11: "NFP/Jobs Miss",
    12: "Unemployment Rate Surprise",
    13: "GDP Beat",
    14: "GDP Miss / Recession Signal",
    15: "Retail Sales Beat",
    16: "Retail Sales Miss",
    17: "PMI/ISM Beat",
    18: "PMI/ISM Miss",
    19: "Trade Tariff / Policy Shock",
    20: "Trade Balance Surprise",
    21: "Debt Ceiling / Fiscal Crisis",
    22: "Fiscal Stimulus / Spending Package",
    23: "US Banking / Financial System Stress",
    24: "Risk-Off Shock",
    25: "Treasury Yield Spike / Bond Market Stress",
    26: "US Political Shock",
    27: "Foreign Central Bank Shock",
    28: "Other / Mixed",
    29: "Trade Policy Reversal Signal",
    30: "Trade Policy Escalation Signal",
    31: "Fed Rate Path Repricing",
    32: "Fiscal Policy Probability Shift",
    33: "Legal/Judicial Event Affecting Economic Policy",
    34: "Legislative Progress or Blockage",
    35: "Executive Policy Signal",
    36: "ECB Rate Change",
    37: "BOJ Rate Change",
    38: "BOE Rate Change",
    39: "Other Major CB Rate Change",
}

In [13]:
ground_truth_csv_path = 'data/dxy_training/claude/gt_test.csv'
max_examples = 3

print(f"📂 Loading few-shot ground truth: {ground_truth_csv_path}")
gt_df = pd.read_csv(ground_truth_csv_path)


required = {"title", "gt_content_type", "gt_event_number", "gt_criticality_level"}
missing = required - set(gt_df.columns)
if missing:
    raise ValueError(
        f"Ground truth CSV is missing required columns: {missing}\n"
        f"Found columns: {list(gt_df.columns)}"
    )

# Derive gt_event_name from gt_event_number via lookup table
gt_df["gt_event_name"] = gt_df["gt_event_number"].map(EVENT_NAMES)
unknown = gt_df["gt_event_name"].isna()
if unknown.any():
    bad = gt_df.loc[unknown, "gt_event_number"].unique().tolist()
    raise ValueError(f"Ground truth contains unrecognised gt_event_number values: {bad}")

# Balanced sample: up to 2 per event type, then cap at max_examples
sampled = (
    gt_df.groupby("gt_event_number", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 2), random_state=42))
    .sample(frac=1, random_state=42)   # shuffle order
    .head(max_examples)
    .reset_index(drop=True)
)

# Resolve content column
content_col = (
    "full_text" if "full_text" in gt_df.columns
    else "content" if "content" in gt_df.columns
    else None
)

examples = []
for _, row in sampled.iterrows():
    if content_col:
        content_preview = str(row.get(content_col, ""))[:600].strip()
    else:
        content_preview = ""   # fall back to title only

    examples.append({
        "title":             str(row["title"]),
        "content_preview":   content_preview,
        "content_type":      str(row["gt_content_type"]),
        # "event_number":      str(row["gt_event_number"]),
        "event_name":        str(row["gt_event_name"]),   # resolved from number
        "criticality_level": str(row["gt_criticality_level"]),
    })

# print(f"✅ {len(examples)} few-shot examples loaded across "
#         f"{sampled['gt_event_number'].nunique()} event types")

📂 Loading few-shot ground truth: data/dxy_training/claude/gt_test.csv


In [10]:
sampled

,id,title,full_text,gt_content_type,gt_criticality_level,gt_event_name
0,2,Fed's Powell says central bank is 'well positi...,Federal Reserve Chair Jerome Powell said Frida...,hard_news,high,Fed Rate Path Repricing
1,5,"US ISM Services PMI falls to 50.8 in March, mi...",The Institute for Supply Management's services...,hard_news,medium,PMI/ISM Miss
2,3,China vows 'fight to the end' as Beijing annou...,China's Ministry of Commerce announced Friday ...,hard_news,high,Trade Policy Escalation Signal


In [16]:
gt = pd.read_csv('data/dxy_training/claude/gt_example.csv')
ft = pd.read_csv('data/dxy_training/gemeni/cnbc_apr_2_4_full_text.csv')

In [18]:
gt.merge(ft[["id", "full_text"]], on="id", how="left").to_csv('data/dxy_training/claude/gt_example_ft.csv', index=False)

In [20]:
april = pd.read_csv('data/dxy_training/claude/cnbc_april.csv').drop(columns = ['Unnamed: 0'])

In [28]:
april[april['full_text'].isna() == False].reset_index(drop=True).to_csv('data/dxy_training/claude/april_cln.csv', index=False)

(324, 13)

In [33]:
pd.read_csv('data/dxy_training/claude/april_results_ind0.csv')

,id,title,link,published,source,query,priority_weight,article_date,prediction_date,fetched_at,...,classification_confidence,event_tier,criticality_level,reasoning,direction,event_name,event_tier_label,is_relevant,is_critical,processed_at
0,0798677b1df2aff47a7df7014fa72de4ce745b87258843...,"US Stock Market Highlights: Dow slides 1,600 p...",https://news.google.com/rss/articles/CBMizgFBV...,"Fri, 04 Apr 2025 07:00:00 GMT",CNBC TV18,"[SOURCES] ""interest rate"" AND ""Federal Reserve...",10,2025-04-03,2025-04-04,2026-02-19T14:44:55.355735,...,high,2,high,This article describes a significant risk-off ...,up,Risk-Off Shock,Tier 2 — Secondary Structural (Risk/Flows/Fiscal),True,True,2026-02-27 03:08:38
